In [2]:
import os
import pandas as pd
from PIL import Image
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib.utils import ImageReader
import io

In [74]:
cranio_results_file = pd.read_csv('/Users/rushil/ichseg/local_results/Rushil_QC_crainotomy_results.csv')
craino_df = pd.DataFrame(cranio_results_file)
#find the index of the highest total_count in the column
highest_index = craino_df['Total_Count'].idxmax()
highest_dataset = 'synthstrip'
print(highest_dataset)

file = pd.read_csv(f'/Users/rushil/ichseg/{highest_dataset}/annotations.csv')
file_df = pd.DataFrame(file)
file_df = file_df[file_df['7 - Craniotomy'] == 'yes']
file_df
def generate_pdf(file_df, output_pdf):
    c = canvas.Canvas(output_pdf, pagesize=landscape(letter))
    width, height = landscape(letter)
    for index, row in file_df.iterrows():
        image = row['Filename']
        image_path = os.path.join('/Users/rushil/ichseg', highest_dataset, 'image_ss_' + highest_dataset, image)
        image = Image.open(image_path)
        image_reader = ImageReader(image)

        c.drawImage(image_reader, 0, 0, width=width, height=height)
        c.showPage()

    c.save()
    
output_pdf_path = f'/Users/rushil/ichseg/{highest_dataset}/only_crainotomy_scans.pdf'
generate_pdf(file_df, output_pdf_path)




synthstrip


In [3]:
no_filelist = ['6153-279_20150605_1251_ct.png', '6153-279_20150606_0754_ct.png', '6153-279_20150609_1306_ct.png', 
            '6153-279_20150702_1607_ct.png', '6155-216_20150606_2103_ct.png', '6155-216_20150607_0417_ct.png', 
            '6155-216_20150609_0408_ct.png', '6225-323_20151029_1938_ct.png', '6155-216_20150608_0123_ct.png',
            '6388-296_20161019_0627_ct.png']

yes_filelist = ['6284-361_20160227_0312_ct.png', '6378-349_20161004_1127_ct.png', '6311-100_20160414_1955_ct.png', 
                '6284-361_20160227_0312_ct.png','6284-361_20160301_0916_ct.png', '6284-361_20160229_0932_ct.png',
                '6101-300_20150109_1657_ct.png']

org_file = pd.read_csv('/Users/rushil/ichseg/synthstrip/annotations.csv')
failure_file = pd.read_csv('/Users/rushil/ichseg/synthstrip/annotations_failures.csv')
org_df = pd.DataFrame(org_file)
failure_df = pd.DataFrame(failure_file)

for file in no_filelist:
    org_df.loc[org_df['Filename'] == file, '7 - Craniotomy'] = 'no'
    failure_df.loc[failure_df['Filename'] == file, '7 - Craniotomy'] = 'no'
    
for file in yes_filelist:
    org_df.loc[org_df['Filename'] == file, '7 - Craniotomy'] = 'yes'
    failure_df.loc[failure_df['Filename'] == file, '7 - Craniotomy'] = 'yes'

org_df.to_csv('/Users/rushil/ichseg/synthstrip/annotations.csv', index=False)
failure_df.to_csv('/Users/rushil/ichseg/synthstrip/annotations_failures.csv', index=False)


In [78]:
synth_crainotomy_images = org_df[org_df['7 - Craniotomy'] == 'yes']['Filename'].tolist()
robust_df = pd.DataFrame(pd.read_csv('/Users/rushil/ichseg/brainchop/annotations.csv'))
robust_df_crainotomy_images = robust_df[robust_df['7 - Craniotomy'] == 'yes']['Filename'].tolist()

robust_df_only_craniotomy_images = set(robust_df_crainotomy_images) - set(synth_crainotomy_images)
print("Images in the robust DataFrame that are not in the original DataFrame:")
for image in robust_df_only_craniotomy_images:
    print(image)
    
print(len(robust_df_crainotomy_images))
print(len(robust_df_only_craniotomy_images))
print(len(synth_crainotomy_images))


Images in the robust DataFrame that are not in the original DataFrame:
6155-216_20150608_0123_ct.png
6388-296_20161019_0627_ct.png
41
2
39


In [4]:
methods = ['v1', 'robust', 'hdctbet', 'ctbet', 'brainchop']

org_file = pd.read_csv('/Users/rushil/ichseg/synthstrip/annotations.csv')
org_df = pd.DataFrame(org_file)
org_craniotomy_images = org_df[org_df['7 - Craniotomy'] == 'yes']['Filename'].tolist()

for method in methods:
    original_csv = pd.read_csv(f"/Users/rushil/ichseg/{method}/annotations.csv")
    failures_csv = pd.read_csv(f"/Users/rushil/ichseg/{method}/annotations_failures.csv")
    original_df = pd.DataFrame(original_csv)
    failures_df = pd.DataFrame(failures_csv)
    
    # Set 'yes' for craniotomy image
    original_df.loc[original_df['Filename'].isin(org_craniotomy_images), '7 - Craniotomy'] = 'yes'
    failures_df.loc[failures_df['Filename'].isin(org_craniotomy_images), '7 - Craniotomy'] = 'yes'
    
    # Set 'no' for all other images
    original_df.loc[~original_df['Filename'].isin(org_craniotomy_images), '7 - Craniotomy'] = 'no'
    failures_df.loc[~failures_df['Filename'].isin(org_craniotomy_images), '7 - Craniotomy'] = 'no'
    
    original_df.to_csv(f"/Users/rushil/ichseg/{method}/annotations.csv", index=False)
    failures_df.to_csv(f"/Users/rushil/ichseg/{method}/annotations_failures.csv", index=False)